# YieldGuard — wafer defect classifier (Colab GPU)

Trains the same `WaferCNN` as `src/models/vision/train.py`, with identical hyperparameters,
so the result drops straight back into the repo.

**Runtime → Change runtime type → T4 GPU** before running. On CPU this is ~3.2 min/epoch;
on a T4 it is a few seconds, so the full 40 epochs finish in well under ten minutes.

Why bother: local training stopped at epoch 5 with macro-F1 0.6550, but `val_loss` was still
falling (1.03 → 0.53) and Scratch precision had collapsed to 0.039 — the sampler over-weights
rare classes early and the model needs more epochs to settle. This is undertrained, not
converged.

In [ ]:
# 1. Upload colab/wm811k_64.npz (14.5 MB) from the repo
from google.colab import files
up = files.upload()          # pick wm811k_64.npz

In [ ]:
# 2. Setup
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, json, time
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

d = np.load("wm811k_64.npz")
# stored as uint8 {0,1,2}; the repo's pipeline feeds float32 in [0,1]
X_train = d["X_train"].astype(np.float32) / 2.0
X_val   = d["X_val"].astype(np.float32) / 2.0
y_train = d["y_train"].astype(np.int64)
y_val   = d["y_val"].astype(np.int64)
CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Local","Random","Scratch","Near-full","None"]
print(X_train.shape, X_val.shape)
for i,c in enumerate(CLASSES): print(f"  {c:<11}{int((y_train==i).sum()):>7}")

In [ ]:
# 3. Model — identical to src/models/vision/model.py
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class WaferCNN(nn.Module):
    def __init__(self, num_classes=9, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1,16,3,1,1,bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = ResBlock(16,32,2); self.layer2 = ResBlock(32,64,2)
        self.layer3 = ResBlock(64,128,2); self.layer4 = ResBlock(128,128,2)
        self.gap = nn.AdaptiveAvgPool2d(1); self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))

print(sum(p.numel() for p in WaferCNN().parameters()), "parameters")

In [ ]:
# 4. Data pipeline — same augmentation, class weights and sampler as the repo
class WaferDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.from_numpy(X).float(); self.y = torch.from_numpy(y).long()
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x, lab = self.X[i], self.y[i]
        if self.augment:
            if torch.rand(1).item() > 0.5: x = x.flip(-1)
            if torch.rand(1).item() > 0.5: x = x.flip(-2)
            k = int(torch.randint(0,4,(1,)).item())
            if k: x = torch.rot90(x, k, dims=[-2,-1])
        return x, lab

counts = np.bincount(y_train, minlength=len(CLASSES)).astype(np.float64)
w = counts.sum() / (len(CLASSES) * np.maximum(counts, 1))
class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)
sample_w = torch.tensor(w[y_train], dtype=torch.double)

BATCH, EPOCHS, LR = 128, 40, 1e-3
train_loader = DataLoader(WaferDataset(X_train,y_train,True), batch_size=BATCH,
    sampler=WeightedRandomSampler(sample_w, len(sample_w), replacement=True),
    num_workers=2, pin_memory=True)
val_loader = DataLoader(WaferDataset(X_val,y_val), batch_size=512,
                        shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 5. Train
def macro_f1(y_true, y_pred, n):
    f1s=[]
    for i in range(n):
        tp=((y_pred==i)&(y_true==i)).sum(); fp=((y_pred==i)&(y_true!=i)).sum(); fn=((y_pred!=i)&(y_true==i)).sum()
        pr=tp/(tp+fp) if tp+fp else 0.0; rc=tp/(tp+fn) if tp+fn else 0.0
        f1s.append(2*pr*rc/(pr+rc) if pr+rc else 0.0)
    return float(np.mean(f1s)), f1s

model = WaferCNN(len(CLASSES)).to(DEVICE)
crit  = nn.CrossEntropyLoss(weight=class_weights)
opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5)

best = 0.0; t0=time.time()
for ep in range(1, EPOCHS+1):
    model.train(); tl=0.0
    for xb,yb in train_loader:
        xb,yb = xb.to(DEVICE,non_blocking=True), yb.to(DEVICE,non_blocking=True)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        tl += loss.item()*len(yb)
    model.eval(); preds=[]; vl=0.0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb); vl += crit(out,yb).item()*len(yb)
            preds.append(out.argmax(1).cpu().numpy())
    p = np.concatenate(preds); f1,_ = macro_f1(y_val, p, len(CLASSES))
    sched.step(f1)
    flag = ""
    if f1 > best:
        best = f1; torch.save(model.state_dict(), "best_model.pt"); flag = "  <- best, saved"
    print(f"Epoch {ep:2d}/{EPOCHS}  train_loss={tl/len(y_train):.4f}  val_loss={vl/len(y_val):.4f}  macro_F1={f1:.4f}{flag}")
print(f"\nBest macro-F1: {best:.4f}   ({time.time()-t0:.0f}s)")

In [ ]:
# 6. Per-class report on the held-out split — the macro average hides where it is weak
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE)); model.eval()
preds=[]
with torch.no_grad():
    for xb,_ in val_loader: preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
p = np.concatenate(preds)
print(f"{'class':<12}{'support':>8}{'prec':>8}{'recall':>8}{'F1':>8}")
print("-"*44); f1s=[]
for i,c in enumerate(CLASSES):
    tp=int(((p==i)&(y_val==i)).sum()); fp=int(((p==i)&(y_val!=i)).sum()); fn=int(((p!=i)&(y_val==i)).sum())
    pr=tp/(tp+fp) if tp+fp else 0.0; rc=tp/(tp+fn) if tp+fn else 0.0
    f1=2*pr*rc/(pr+rc) if pr+rc else 0.0; f1s.append(f1)
    print(f"{c:<12}{int((y_val==i).sum()):>8}{pr:>8.3f}{rc:>8.3f}{f1:>8.3f}")
print("-"*44)
print(f"{'MACRO-F1':<12}{'':>24}{np.mean(f1s):>8.4f}")
print(f"{'accuracy':<12}{'':>24}{(p==y_val).mean():>8.4f}   <- inflated by the 'None' class, do not quote it")

In [ ]:
# 7. Download best_model.pt -> put it in src/models/vision/checkpoints/best_model.pt
from google.colab import files
files.download("best_model.pt")

## Back in the repo

```bash
mv ~/Downloads/best_model.pt src/models/vision/checkpoints/best_model.pt
.venv/bin/python src/mcp_server/test_server.py
.venv/bin/python src/eval/run_eval.py
```

Paste the per-class table from cell 6 into `src/models/vision/NOTES.md`. Quote **macro-F1**,
never accuracy — the `None` class is 59% of the validation split, so accuracy flatters a model
that has learned nothing.